# Mosaic eval harness — GPU run on Colab

Runs `src/eval/{shortcut_check,robustness,calibration,error_analysis}.py` with `--device cuda`, against CIFAKE + SID-Set + WildFake combined (via `scan_all_sources()` in `src/eval/data_compat.py`).

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU.

**What you need on hand:**
- `mosaic_code.zip` (src/, configs/, requirements.txt, pyproject.toml, plus the already-cleaned SID-Set/WildFake manifests — no raw images, no checkpoint)
- Your checkpoint `.pt` file (e.g. `wildfake_head.pt`, or `model_best.pt`)
- A Kaggle API token (`kaggle.json`) to fetch CIFAKE — from https://www.kaggle.com/settings -> API -> Create New Token
- **For WildFake only**: you must open https://modelscope.cn/datasets/hy2628982280/WildFake/summary in a browser and click "translate" once, per the challenge instructions, before downloading — this can't be scripted. Then download the 5 focused-subset archives yourself: `Images/Real/{afhq,celebahq,church}.zip` and `Images/Diffusion_based/{DDIM,DDPM}.zip`.

## 1. Upload the code

In [ ]:
from google.colab import files

print("Select mosaic_code.zip")
uploaded = files.upload()
assert any(name.endswith(".zip") for name in uploaded), "Expected a .zip file"
zip_name = [name for name in uploaded if name.endswith(".zip")][0]

In [ ]:
import shutil

shutil.unpack_archive(zip_name, "/content/mosaic")
%cd /content/mosaic
!ls

## 2. Install dependencies

Colab's preinstalled torch already includes CUDA support for Linux, so a normal `pip install -r requirements.txt` picks up a CUDA-enabled torch build without any special index URL. Then install the project itself in editable mode so `import src.xxx` resolves regardless of invocation style (mirrors the local `pip install -e .` fix).

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -20
!pip install -q -e . 2>&1 | tail -20

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — check Runtime > Change runtime type > GPU")

## 3. Upload the checkpoint

In [ ]:
import os
os.makedirs("outputs/baseline", exist_ok=True)

print("Select your checkpoint .pt file")
uploaded = files.upload()
ckpt_name = list(uploaded)[0]
shutil.move(ckpt_name, "outputs/baseline/model_best.pt")
!ls -la outputs/baseline/

## 4a. Fetch CIFAKE (downloaded fresh here — faster than uploading 469MB from a laptop)

Reproduces the same `data/raw/CIFAKE/{train,test}/{REAL,FAKE}` layout used locally (native Kaggle shape, matching `src/eval/data_compat.py`'s `scan_cifake_nested`).

In [ ]:
print("Select your kaggle.json API token")
uploaded = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
import kagglehub

cifake_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print("Downloaded to:", cifake_path)
!ls "{cifake_path}"

In [ ]:
# Symlink kagglehub's cache into data/raw/CIFAKE/{train,test} — same trick used locally,
# avoids copying ~120K files.
os.makedirs("data/raw/CIFAKE", exist_ok=True)
for split in ("train", "test"):
    src = os.path.join(cifake_path, split)
    dst = f"data/raw/CIFAKE/{split}"
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
!find data/raw/CIFAKE -maxdepth 2
!find data/raw/CIFAKE -type f | wc -l

## 4b. Fetch SID-Set

Downloads the Parquet shards to `data/raw/sid_set` (matching `train_clip_head.py`'s `SID_SET_RAW_ROOT`). The clean manifest referencing them (`data/processed/sid_set/clean_manifest.csv`) already came in with `mosaic_code.zip` — Vicky's cleaner already ran, this just needs the raw bytes it points at.

In [ ]:
from huggingface_hub import snapshot_download

sid_set_dir = snapshot_download(repo_id="saberzl/SID_Set", repo_type="dataset", local_dir="data/raw/sid_set")
print("Downloaded to:", sid_set_dir)
!find data/raw/sid_set -maxdepth 2

## 4c. Fetch WildFake (manual step required first)

ModelScope's translate-page requirement means this can't be automated — after translating the page and downloading the 5 archives yourself (see intro cell), upload them here. They can be large; if `files.upload()` is too slow for your connection, mount Google Drive instead (commented alternative below) and copy them from there.

Archive paths must match `clean_wildfake.py`'s `DEFAULT_ARCHIVES` exactly, relative to `data/raw/wildfake/`:
`Images/Real/{afhq,celebahq,church}.zip`, `Images/Diffusion_based/{DDIM,DDPM}.zip`.

In [ ]:
import os

os.makedirs("data/raw/wildfake/Images/Real", exist_ok=True)
os.makedirs("data/raw/wildfake/Images/Diffusion_based", exist_ok=True)

# --- Option A: direct upload (fine for smaller files / fast connections) ---
print("Select afhq.zip, celebahq.zip, church.zip, DDIM.zip, DDPM.zip (multi-select in the file picker)")
uploaded = files.upload()
REAL_NAMES = {"afhq.zip", "celebahq.zip", "church.zip"}
for name in uploaded:
    dest = f"data/raw/wildfake/Images/Real/{name}" if name in REAL_NAMES else f"data/raw/wildfake/Images/Diffusion_based/{name}"
    shutil.move(name, dest)

# --- Option B: Google Drive instead (uncomment if uploads are too slow) ---
# from google.colab import drive
# drive.mount("/content/drive")
# for name in ("afhq.zip", "celebahq.zip", "church.zip"):
#     shutil.copy(f"/content/drive/MyDrive/wildfake/{name}", f"data/raw/wildfake/Images/Real/{name}")
# for name in ("DDIM.zip", "DDPM.zip"):
#     shutil.copy(f"/content/drive/MyDrive/wildfake/{name}", f"data/raw/wildfake/Images/Diffusion_based/{name}")

!find data/raw/wildfake -maxdepth 3

## 5. Run the eval scripts on GPU

Each accepts `--device` — pass `cuda` here instead of the `cpu` default used locally. `shortcut_check.py` first, per the project's priority order (early warning on shortcut learning before polishing the rest).

In [ ]:
!python src/eval/shortcut_check.py --checkpoint outputs/baseline/model_best.pt --device cuda

In [ ]:
!python src/eval/robustness.py --checkpoint outputs/baseline/model_best.pt --device cuda --out outputs/robustness_table.csv

In [ ]:
!python src/eval/calibration.py --checkpoint outputs/baseline/model_best.pt --device cuda

In [ ]:
!python src/eval/error_analysis.py --checkpoint outputs/baseline/model_best.pt --device cuda --out outputs/error_analysis.md

## 6. Download the results back to your laptop

In [ ]:
shutil.make_archive("/content/mosaic_outputs", "zip", "outputs")
files.download("/content/mosaic_outputs.zip")